# Коварная домашка

## С противным решением

## Что делаю в двух словах

1. Стараюсь фиксировать seed `993`, но не совсем так как необходимо
2. Код делит исходные изображения на train/valid.
4. Создаю аугментированный датасет под формат YOLO.
5. Имитирую внешний вид test: грязь, пятна, артефакты, blur, JPEG/downscale-деградацию и шум.
6. Обучаю `YOLO26m` на уже испорченных изображениях.
7. Визуализируею предсказания на test.
8. Формирую итоговый файл `sample_submission.csv` в Colab.

## Главная идея решения

Test-изображения в задаче выглядят сильно испорченными: blur, снижение качества, шумы и грязные артефакты. Поэтому модель обучается не на чистых исходниках, а на специально материализованном аугментированном датасете. Так требует сама задача, это немножко странно, но иначе один промт гптишки и все равны

Важная последовательность аугментаций:

> сначала накладывается dirty-эффект, затем изображение дополнительно портится через downscale/JPEG/blur/noise.

Это ближе к визуальному виду test-выборки и помогает модели устойчивее работать на плохом качестве изображений, но все равно не совсем то, что я хотел

Я понимаю, что решение с формированием датасета тем образом, который выбрал я — это очень непрофессионально и тупо, НО простите, пожалуйста!

В конце концов, это же никак не влияет на последующую работу модельки, а просто некоторые проблемы в трейне

## 0. Установка зависимостей для Google Colab

В Colab окружение каждый раз создаётся заново, поэтому в начале ноутбука ставим/обновляем основные библиотеки:

- `ultralytics` — обучение и инференс YOLO;
- `albumentations` — кастомные аугментации;
- `opencv-python-headless` — работа с изображениями без GUI-зависимостей.

После установки в Colab иногда полезно сделать `Runtime → Restart session`, если среда попросит перезапуск.


In [ ]:
# ============================================================
# 0. Install dependencies for Google Colab
# ============================================================

!pip -q install -U ultralytics albumentations opencv-python-headless


## 1. Импорты

Здесь подключаются все библиотеки, которые используются дальше:

- `cv2`, `numpy`, `pandas` — работа с изображениями, массивами и таблицами;
- `albumentations` — кастомный пайплайн аугментаций;
- `torch` и `ultralytics.YOLO` — обучение и инференс модели;
- `Path`, `shutil`, `yaml`, `hashlib` — организация файлов, путей, конфигов и воспроизводимых разбиений.


Также отключаю варнинги

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

import os
import sys
import cv2
import yaml
import math
import time
import shutil
import random
import hashlib
import inspect
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import albumentations as A
from ultralytics import YOLO
from IPython.display import display

warnings.filterwarnings("ignore")


## 2. Глобальная конфигурация под Google Colab

Эта ячейка заменяет Kaggle-зависимые пути на Colab-пути.

Что она делает:

- при необходимости монтирует Google Drive;
- ищет датасет в `/content` и в типичных папках Google Drive;
- умеет автоматически распаковать `2026-cv-competition.zip`, если архив лежит в `/content` или в `MyDrive`;
- создаёт рабочую папку `/content/working`;
- настраивает обучение так, чтобы при наличии двух GPU использовать `device=[0, 1]`.

Ожидаемая структура датасета после распаковки/подключения может быть обычной:

```text
2026-cv-competition/
├── train/train/images
├── train/train/labels
├── test/test/images
└── sample_submission.csv
```

Или вложенной, как в твоём архиве:

```text
2026-cv-competition.zip
└── 2026-cv-competition/
    └── 2026-cv-competition/
        ├── train/train/images
        ├── train/train/labels
        ├── test/test/images
        └── sample_submission.csv
```

Код ниже специально учитывает этот вариант и выбирает внутреннюю папку `2026-cv-competition`, где лежат `train`, `test` и `sample_submission.csv`.

Если у тебя датасет лежит в другом месте, просто поменяй `DATASET_DIR_MANUAL`.


In [ ]:
# ============================================================
# 2. Global config for Google Colab
# ============================================================

from pathlib import Path

# ============================================================
# Reproducibility
# ============================================================

SEED = 993

# ============================================================
# Task parameters
# ============================================================

NUM_CLASSES = 52
IMG_SIZE = 640

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# ============================================================
# Google Colab paths
# ============================================================

# В Colab основная рабочая директория — /content.
COLAB_ROOT = Path("/content")
WORK_DIR = COLAB_ROOT / "working"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Если датасет лежит на Google Drive, оставь True.
# Если ты вручную загрузил/распаковал датасет прямо в /content, можно поставить False.
MOUNT_GOOGLE_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive")

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Google Drive mounted:", DRIVE_ROOT)
    except Exception as e:
        print("Google Drive не смонтирован. Если датасет уже в /content, это не проблема.")
        print("Причина:", repr(e))

# ============================================================
# Dataset location
# ============================================================

DATASET_NAME = "2026-cv-competition"

# Если датасет уже распакован в конкретную папку, можешь явно указать путь:
# DATASET_DIR_MANUAL = Path("/content/drive/MyDrive/2026-cv-competition")
# DATASET_DIR_MANUAL = Path("/content/2026-cv-competition")
DATASET_DIR_MANUAL = None

# Если датасет хранится архивом, можешь явно указать путь:
# DATASET_ZIP_PATH = Path("/content/drive/MyDrive/2026-cv-competition.zip")
# DATASET_ZIP_PATH = Path("/content/2026-cv-competition.zip")
DATASET_ZIP_PATH = None

# Автоматические места, где ноутбук будет искать архив.
AUTO_ZIP_CANDIDATES = [
    # Архив прямо в /content.
    COLAB_ROOT / f"{DATASET_NAME}.zip",

    # Архив лежит в папке /content/2026-cv-competition/.
    COLAB_ROOT / DATASET_NAME / f"{DATASET_NAME}.zip",

    # Архив прямо в Google Drive.
    DRIVE_ROOT / f"{DATASET_NAME}.zip",

    # Архив лежит в папке Google Drive/2026-cv-competition/.
    DRIVE_ROOT / DATASET_NAME / f"{DATASET_NAME}.zip",

    # Частые варианты с папками datasets/kaggle/Kaggle.
    DRIVE_ROOT / "datasets" / f"{DATASET_NAME}.zip",
    DRIVE_ROOT / "datasets" / DATASET_NAME / f"{DATASET_NAME}.zip",
    DRIVE_ROOT / "kaggle" / f"{DATASET_NAME}.zip",
    DRIVE_ROOT / "kaggle" / DATASET_NAME / f"{DATASET_NAME}.zip",
    DRIVE_ROOT / "Kaggle" / f"{DATASET_NAME}.zip",
    DRIVE_ROOT / "Kaggle" / DATASET_NAME / f"{DATASET_NAME}.zip",
]

# Автоматические места, где ноутбук будет искать уже распакованный датасет.
AUTO_DATASET_DIR_CANDIDATES = [
    # Самый важный случай: после распаковки получается
    # /content/2026-cv-competition/2026-cv-competition/...
    COLAB_ROOT / DATASET_NAME / DATASET_NAME,

    # Обычный случай: /content/2026-cv-competition/...
    COLAB_ROOT / DATASET_NAME,

    # Если архив распаковали вручную в /content/dataset или /content/data.
    COLAB_ROOT / "dataset" / DATASET_NAME,
    COLAB_ROOT / "dataset",
    COLAB_ROOT / "data" / DATASET_NAME,
    COLAB_ROOT / "data",

    # Те же варианты на Google Drive.
    DRIVE_ROOT / DATASET_NAME / DATASET_NAME,
    DRIVE_ROOT / DATASET_NAME,
    DRIVE_ROOT / "datasets" / DATASET_NAME / DATASET_NAME,
    DRIVE_ROOT / "datasets" / DATASET_NAME,
    DRIVE_ROOT / "kaggle" / DATASET_NAME / DATASET_NAME,
    DRIVE_ROOT / "kaggle" / DATASET_NAME,
    DRIVE_ROOT / "Kaggle" / DATASET_NAME / DATASET_NAME,
    DRIVE_ROOT / "Kaggle" / DATASET_NAME,
]


def maybe_extract_dataset_zip():
    """Распаковывает zip с датасетом в /content, если архив найден."""
    zip_path = DATASET_ZIP_PATH

    if zip_path is None:
        zip_path = next((p for p in AUTO_ZIP_CANDIDATES if p.exists()), None)
    else:
        zip_path = Path(zip_path)

    if zip_path is None or not zip_path.exists():
        return None

    print("Найден архив датасета:", zip_path)
    print("Распаковываю в:", COLAB_ROOT)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(COLAB_ROOT)

    print("Распаковка завершена.")
    return zip_path


def collect_images_quick(images_dir):
    images_dir = Path(images_dir)
    if not images_dir.exists():
        return []

    return sorted([
        p for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])


def find_yolo_image_label_dirs_in_root(root):
    root = Path(root)
    if not root.exists():
        return None, None

    # Сначала проверяем самые вероятные структуры.
    exact_pairs = [
        # Вложенная структура: root/2026-cv-competition/train/train/images
        (root / DATASET_NAME / "train" / "train" / "images", root / DATASET_NAME / "train" / "train" / "labels"),

        # Нормальная структура соревнования.
        (root / "train" / "train" / "images", root / "train" / "train" / "labels"),
        (root / "train" / "images", root / "train" / "labels"),
        (root / "images" / "train", root / "labels" / "train"),
        (root / "images", root / "labels"),
    ]

    for images_dir, labels_dir in exact_pairs:
        if images_dir.exists() and labels_dir.exists() and len(collect_images_quick(images_dir)) > 0:
            return images_dir, labels_dir

    # Потом более общий поиск.
    candidates = []
    for images_dir in root.rglob("images"):
        if not images_dir.is_dir():
            continue

        labels_dir = images_dir.parent / "labels"
        if not labels_dir.exists():
            continue

        image_count = len(collect_images_quick(images_dir))
        if image_count == 0:
            continue

        path_text = str(images_dir).lower().replace("\\", "/")
        priority = 0

        if "/train/" in path_text or path_text.endswith("/train/images"):
            priority += 100
        if "valid" in path_text or "val" in path_text or "test" in path_text:
            priority -= 50

        candidates.append((priority, image_count, images_dir, labels_dir))

    if len(candidates) == 0:
        return None, None

    candidates = sorted(candidates, key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0][2], candidates[0][3]


def find_sample_submission_in_root(root):
    root = Path(root)
    if not root.exists():
        return None

    exact_candidates = [
        root / "sample_submission.csv",
        root / DATASET_NAME / "sample_submission.csv",
    ]

    for exact in exact_candidates:
        if exact.exists():
            return exact

    candidates = sorted(root.rglob("sample_submission.csv"))
    if len(candidates) == 0:
        return None

    return candidates[0]


def find_test_images_in_root(root):
    root = Path(root)
    if not root.exists():
        return None

    exact_dirs = [
        # Вложенная структура: root/2026-cv-competition/test/test/images
        root / DATASET_NAME / "test" / "test" / "images",

        # Нормальная структура соревнования.
        root / "test" / "test" / "images",
        root / "test" / "images",
        root / "images" / "test",
    ]

    for images_dir in exact_dirs:
        if images_dir.exists() and len(collect_images_quick(images_dir)) > 0:
            return images_dir

    candidates = []
    for images_dir in root.rglob("images"):
        if not images_dir.is_dir():
            continue

        image_count = len(collect_images_quick(images_dir))
        if image_count == 0:
            continue

        path_text = str(images_dir).lower().replace("\\", "/")
        priority = 0

        if "test" in path_text:
            priority += 100
        if "train" in path_text:
            priority -= 50
        if "valid" in path_text or "val" in path_text:
            priority -= 30

        candidates.append((priority, image_count, images_dir))

    if len(candidates) == 0:
        return None

    candidates = sorted(candidates, key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0][2]


def expand_possible_dataset_roots(root):
    """
    Возвращает варианты корня датасета от более конкретных к более общим.

    Это нужно для случая:
        2026-cv-competition/
        └── 2026-cv-competition/
            ├── train/
            ├── test/
            └── sample_submission.csv

    Если передали внешнюю папку, сначала проверяем внутреннюю.
    """
    root = Path(root)

    possible_roots = [
        root / DATASET_NAME / DATASET_NAME,
        root / DATASET_NAME,
        root,
    ]

    unique_roots = []
    seen = set()

    for candidate in possible_roots:
        candidate = candidate.resolve() if candidate.exists() else candidate
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_roots.append(candidate)

    return unique_roots


def dataset_root_is_complete(root):
    """Проверяет, что в корне можно найти train/images, train/labels, test/images и sample_submission.csv."""
    images_dir, labels_dir = find_yolo_image_label_dirs_in_root(root)
    sample_path = find_sample_submission_in_root(root)
    test_dir = find_test_images_in_root(root)

    return (
        images_dir is not None
        and labels_dir is not None
        and sample_path is not None
        and test_dir is not None
    )


def resolve_dataset_dir():
    """Находит внутреннюю корневую папку соревнования."""
    if DATASET_DIR_MANUAL is not None:
        manual_dir = Path(DATASET_DIR_MANUAL)
        if not manual_dir.exists():
            raise FileNotFoundError(f"DATASET_DIR_MANUAL не существует: {manual_dir}")

        for possible_root in expand_possible_dataset_roots(manual_dir):
            if possible_root.exists() and dataset_root_is_complete(possible_root):
                return possible_root

        raise FileNotFoundError(
            "DATASET_DIR_MANUAL существует, но внутри не найдены train/test/sample_submission.csv. "
            f"Проверь путь: {manual_dir}"
        )

    maybe_extract_dataset_zip()

    # Сначала проверяем явные кандидаты. Для каждого кандидата отдельно
    # пробуем внутреннюю папку 2026-cv-competition, а уже потом саму папку.
    for candidate in AUTO_DATASET_DIR_CANDIDATES:
        for possible_root in expand_possible_dataset_roots(candidate):
            if not possible_root.exists():
                continue

            if dataset_root_is_complete(possible_root):
                return possible_root

    # Запасной вариант: если архив распаковался необычно, ищем sample_submission.csv внутри /content.
    # Важно: не сканируем /content/drive целиком, иначе Google Drive может очень долго обходиться.
    sample_candidates = []
    for current_root, dirs, files in os.walk(COLAB_ROOT):
        dirs[:] = [d for d in dirs if d != "drive" and not d.startswith(".")]

        if "sample_submission.csv" in files:
            sample_candidates.append(Path(current_root) / "sample_submission.csv")

    for sample_path in sorted(sample_candidates):
        # Обычно именно parent у sample_submission.csv является правильным внутренним корнем.
        candidate = sample_path.parent

        for possible_root in expand_possible_dataset_roots(candidate):
            if possible_root.exists() and dataset_root_is_complete(possible_root):
                return possible_root

    raise FileNotFoundError(
        "Не удалось найти датасет.\n"
        "Ожидаемый вариант для Colab:\n"
        "  /content/2026-cv-competition/2026-cv-competition/train/train/images\n"
        "  /content/2026-cv-competition/2026-cv-competition/train/train/labels\n"
        "  /content/2026-cv-competition/2026-cv-competition/test/test/images\n"
        "  /content/2026-cv-competition/2026-cv-competition/sample_submission.csv\n"
        "Или положи архив 2026-cv-competition.zip в /content, /content/2026-cv-competition, "
        "Google Drive, Google Drive/2026-cv-competition, либо явно задай DATASET_DIR_MANUAL."
    )


DATASET_DIR = resolve_dataset_dir()
DATASET_SEARCH_ROOT = DATASET_DIR

RAW_TRAIN_IMAGES_DIR, RAW_TRAIN_LABELS_DIR = find_yolo_image_label_dirs_in_root(DATASET_DIR)
TEST_IMAGES_DIR = find_test_images_in_root(DATASET_DIR)
SAMPLE_SUBMISSION_PATH = find_sample_submission_in_root(DATASET_DIR)

assert RAW_TRAIN_IMAGES_DIR is not None, f"Не найдена папка train images внутри {DATASET_DIR}"
assert RAW_TRAIN_LABELS_DIR is not None, f"Не найдена папка train labels внутри {DATASET_DIR}"
assert TEST_IMAGES_DIR is not None, f"Не найдена папка test images внутри {DATASET_DIR}"
assert SAMPLE_SUBMISSION_PATH is not None, f"Не найден sample_submission.csv внутри {DATASET_DIR}"

# Эти переменные нужны дальше в старом resolver-блоке.
# Оставляем их явно, чтобы остальной код не пришлось переписывать.
MANUAL_TRAIN_IMAGES_DIR = RAW_TRAIN_IMAGES_DIR
MANUAL_TRAIN_LABELS_DIR = RAW_TRAIN_LABELS_DIR
MANUAL_TEST_IMAGES_DIR = TEST_IMAGES_DIR
MANUAL_SAMPLE_SUBMISSION_PATH = SAMPLE_SUBMISSION_PATH

print("Dataset dir:", DATASET_DIR)
print("Train images:", RAW_TRAIN_IMAGES_DIR)
print("Train labels:", RAW_TRAIN_LABELS_DIR)
print("Test images:", TEST_IMAGES_DIR)
print("Sample submission:", SAMPLE_SUBMISSION_PATH)

# Проверяем, что внутри действительно есть нужные файлы.
train_image_paths = sorted([
    p for p in RAW_TRAIN_IMAGES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
])

train_label_paths = sorted([
    p for p in RAW_TRAIN_LABELS_DIR.iterdir()
    if p.is_file() and p.suffix.lower() == ".txt"
])

test_image_paths = sorted([
    p for p in TEST_IMAGES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
])

print()
print("Train images count:", len(train_image_paths))
print("Train labels count:", len(train_label_paths))
print("Test images count:", len(test_image_paths))

assert len(train_image_paths) > 0, "В train/images не найдено изображений"
assert len(train_label_paths) > 0, "В train/labels не найдено .txt файлов разметки"
assert len(test_image_paths) > 0, "В test/images не найдено изображений"

# ============================================================
# Train/valid split
# ============================================================

VALID_FRACTION = 0.15

# ============================================================
# Offline augmented YOLO dataset
# ============================================================

OFFLINE_ROOT = WORK_DIR / "yolo26m_offline_aug_dataset"
OFFLINE_DATA_YAML = OFFLINE_ROOT / "data.yaml"

OFFLINE_TRAIN_IMAGES_DIR = OFFLINE_ROOT / "images" / "train"
OFFLINE_TRAIN_LABELS_DIR = OFFLINE_ROOT / "labels" / "train"

OFFLINE_VALID_IMAGES_DIR = OFFLINE_ROOT / "images" / "valid"
OFFLINE_VALID_LABELS_DIR = OFFLINE_ROOT / "labels" / "valid"

# Так как test сильно искажен, то я не добавляю оригиналы.
KEEP_ORIGINAL_TRAIN_IMAGES = False
KEEP_ORIGINAL_VALID_IMAGES = False

# Сколько offline-аугментированных копий делать.
TRAIN_AUG_COPIES_PER_IMAGE = 3
VALID_AUG_COPIES_PER_IMAGE = 1

MAX_IMAGES_DEBUG = None

# Validation тоже делаем похожим на test.
USE_DIRTY_VALID = True

# ============================================================
# YOLO26m training
# ============================================================

MODEL_WEIGHTS = "yolo26m.pt"

TRAIN_EPOCHS = 30

# В Colab обычно дают одну T4, но если доступно 2 GPU, используем обе.
AUTO_USE_TWO_GPUS = True
NUM_AVAILABLE_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

if AUTO_USE_TWO_GPUS and NUM_AVAILABLE_GPUS >= 2:
    TRAIN_DEVICE = [0, 1]
    BATCH_SIZE = 8
else:
    TRAIN_DEVICE = 0 if torch.cuda.is_available() else "cpu"
    BATCH_SIZE = 4

# Для Colab безопаснее держать workers небольшим.
WORKERS = 2

PROJECT_DIR = str(WORK_DIR / "yolo26m_runs")
RUN_NAME = "solution_yolo26m_img640_no_yolo_aug_colab"

print()
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", NUM_AVAILABLE_GPUS)
for i in range(NUM_AVAILABLE_GPUS):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("TRAIN_DEVICE:", TRAIN_DEVICE)
print("BATCH_SIZE:", BATCH_SIZE)

# ============================================================
# Submission
# ============================================================

SUBMISSION_PATH = WORK_DIR / "sample_submission.csv"


## 3. Воспроизводимость эксперимента

Эта ячейка фиксирует случайность в `random`, `numpy` и `torch`. Не совсем так, как требовалось в условиях, но в любом случае, я думаю этого достаточно

Дополнительно включается более детерминированный режим CUDA/cuDNN, насколько это возможно в рамках PyTorch и Colab-среды.


In [ ]:
# ============================================================
# 3. Reproducibility
# ============================================================

def seed_everything(seed=SEED, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(bool(deterministic), warn_only=True)
    except Exception:
        pass

    try:
        torch.backends.cudnn.deterministic = bool(deterministic)
        torch.backends.cudnn.benchmark = not bool(deterministic)
    except Exception:
        pass

    return seed


seed_everything(SEED, deterministic=True)
print("Seed fixed:", SEED)
print("CUDA available:", torch.cuda.is_available())


## 4. Вспомогательные функции

В этом блоке находятся небольшие служебные функции, которые переиспользуются дальше по ноутбуку.

Они отвечают за:

- стабильный hash для детерминированных операций;
- приведение изображений к `uint8`;
- поиск изображений в папках;
- чтение и сохранение YOLO-разметки;
- конвертации и безопасную работу с bounding boxes.

Такие функции вынесены отдельно, чтобы основной пайплайн ниже был чище и легче читался.


In [ ]:
# ============================================================
# 4. Utility helpers
# ============================================================

def stable_int_hash(text, seed=SEED):
    s = f"{seed}_{text}".encode("utf-8")
    return int(hashlib.md5(s).hexdigest(), 16) % (2**32 - 1)


def as_uint8(img):
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    return img


def collect_images(images_dir):
    images_dir = Path(images_dir)
    paths = []

    for p in images_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            paths.append(p)

    return sorted(paths)


def read_image_rgb(path):
    path = Path(path)
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def save_image_rgb_jpg(path, image_rgb, quality=96):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.cvtColor(as_uint8(image_rgb), cv2.COLOR_RGB2BGR)
    ok = cv2.imwrite(
        str(path),
        image_bgr,
        [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)],
    )

    if not ok:
        raise RuntimeError(f"Не удалось сохранить изображение: {path}")


def clip_yolo_bbox(xc, yc, bw, bh):
    xc, yc, bw, bh = map(float, [xc, yc, bw, bh])

    x1 = xc - bw / 2
    y1 = yc - bh / 2
    x2 = xc + bw / 2
    y2 = yc + bh / 2

    x1 = max(0.0, min(1.0, x1))
    y1 = max(0.0, min(1.0, y1))
    x2 = max(0.0, min(1.0, x2))
    y2 = max(0.0, min(1.0, y2))

    if x2 <= x1 or y2 <= y1:
        return None

    return [
        (x1 + x2) / 2,
        (y1 + y2) / 2,
        x2 - x1,
        y2 - y1,
    ]


def read_yolo_label(label_path):
    label_path = Path(label_path)
    bboxes, labels = [], []

    if not label_path.exists():
        return bboxes, labels

    with open(label_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        try:
            class_id = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
        except Exception:
            continue

        if not (0 <= class_id < NUM_CLASSES):
            continue

        clipped = clip_yolo_bbox(xc, yc, bw, bh)
        if clipped is None:
            continue

        bboxes.append(clipped)
        labels.append(class_id)

    return bboxes, labels


def write_yolo_label(label_path, bboxes, labels):
    label_path = Path(label_path)
    label_path.parent.mkdir(parents=True, exist_ok=True)

    lines = []
    for bbox, label in zip(bboxes, labels):
        clipped = clip_yolo_bbox(*bbox)
        if clipped is None:
            continue

        xc, yc, bw, bh = clipped
        lines.append(f"{int(label)} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}\n")

    with open(label_path, "w", encoding="utf-8") as f:
        f.writelines(lines)


## 4.1. EDA: анализ датасета и объектов

Этот блок ничего не меняет в обучении и инференсе. Он только читает исходные изображения и YOLO-разметку, собирает статистику по датасету и строит базовые графики:

- сколько изображений, label-файлов и объектов есть в train/test;
- есть ли изображения без разметки или label-файлы без соответствующей картинки;
- распределение количества объектов на изображение;
- распределение объектов по 52 классам;
- размеры bbox, площади bbox и положение центров объектов;
- примеры изображений с большим количеством объектов.



In [ ]:
# ============================================================
# 4.1. Exploratory Data Analysis / EDA
# ============================================================

from collections import Counter

# Для EDA используем только исходный train/test, ничего не перезаписываем
# и не меняем переменные, которые дальше нужны для обучения.
EDA_IMAGES_DIR = Path(RAW_TRAIN_IMAGES_DIR)
EDA_LABELS_DIR = Path(RAW_TRAIN_LABELS_DIR)
EDA_TEST_IMAGES_DIR = Path(TEST_IMAGES_DIR)

eda_image_paths = collect_images(EDA_IMAGES_DIR)
eda_test_image_paths = collect_images(EDA_TEST_IMAGES_DIR) if EDA_TEST_IMAGES_DIR.exists() else []
eda_label_paths = sorted([
    p for p in EDA_LABELS_DIR.iterdir()
    if p.is_file() and p.suffix.lower() == ".txt"
])

print("EDA train images dir:", EDA_IMAGES_DIR)
print("EDA train labels dir:", EDA_LABELS_DIR)
print("EDA test images dir:", EDA_TEST_IMAGES_DIR)
print()
print("Train images:", len(eda_image_paths))
print("Train label files:", len(eda_label_paths))
print("Test images:", len(eda_test_image_paths))


def parse_yolo_label_for_eda(label_path):
    """
    Читает YOLO-разметку и дополнительно считает проблемные строки.

    Ожидаемый формат строки:
        class_id x_center y_center width height

    Координаты нормализованы в диапазон [0, 1].
    """
    label_path = Path(label_path)

    bboxes = []
    labels = []
    audit = {
        "missing_label_file": 0,
        "empty_label_file": 0,
        "raw_lines": 0,
        "valid_objects": 0,
        "malformed_lines": 0,
        "invalid_class_lines": 0,
        "invalid_bbox_lines": 0,
    }

    if not label_path.exists():
        audit["missing_label_file"] = 1
        return bboxes, labels, audit

    with open(label_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    audit["raw_lines"] = len(lines)
    if len(lines) == 0:
        audit["empty_label_file"] = 1
        return bboxes, labels, audit

    for line in lines:
        parts = line.split()

        if len(parts) < 5:
            audit["malformed_lines"] += 1
            continue

        try:
            class_id = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
        except Exception:
            audit["malformed_lines"] += 1
            continue

        if not (0 <= class_id < NUM_CLASSES):
            audit["invalid_class_lines"] += 1
            continue

        clipped = clip_yolo_bbox(xc, yc, bw, bh)
        if clipped is None:
            audit["invalid_bbox_lines"] += 1
            continue

        bboxes.append(clipped)
        labels.append(class_id)
        audit["valid_objects"] += 1

    return bboxes, labels, audit


eda_image_rows = []
eda_object_rows = []
eda_audit_rows = []

for image_path in tqdm(eda_image_paths, desc="EDA: reading train images and labels"):
    image_path = Path(image_path)
    label_path = EDA_LABELS_DIR / f"{image_path.stem}.txt"

    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        height, width = np.nan, np.nan
        image_read_ok = False
    else:
        height, width = image_bgr.shape[:2]
        image_read_ok = True

    bboxes, labels, audit = parse_yolo_label_for_eda(label_path)

    eda_image_rows.append({
        "image_id": image_path.stem,
        "file_name": image_path.name,
        "image_path": str(image_path),
        "label_path": str(label_path),
        "image_read_ok": image_read_ok,
        "width": width,
        "height": height,
        "resolution": f"{int(width)}x{int(height)}" if image_read_ok else "unreadable",
        "has_label_file": label_path.exists(),
        "n_objects": len(labels),
    })

    audit_row = {"image_id": image_path.stem, "file_name": image_path.name}
    audit_row.update(audit)
    eda_audit_rows.append(audit_row)

    for bbox, class_id in zip(bboxes, labels):
        xc, yc, bw, bh = bbox
        x_min = max(0.0, xc - bw / 2)
        y_min = max(0.0, yc - bh / 2)
        x_max = min(1.0, xc + bw / 2)
        y_max = min(1.0, yc + bh / 2)

        if image_read_ok:
            box_width_px = bw * width
            box_height_px = bh * height
            box_area_px = box_width_px * box_height_px
        else:
            box_width_px = np.nan
            box_height_px = np.nan
            box_area_px = np.nan

        eda_object_rows.append({
            "image_id": image_path.stem,
            "file_name": image_path.name,
            "class_id": int(class_id),
            "x_center": xc,
            "y_center": yc,
            "bbox_width_norm": bw,
            "bbox_height_norm": bh,
            "bbox_area_norm": bw * bh,
            "bbox_aspect_ratio": bw / max(bh, 1e-9),
            "x_min": x_min,
            "y_min": y_min,
            "x_max": x_max,
            "y_max": y_max,
            "bbox_width_px": box_width_px,
            "bbox_height_px": box_height_px,
            "bbox_area_px": box_area_px,
        })

eda_images_df = pd.DataFrame(eda_image_rows)
eda_objects_df = pd.DataFrame(eda_object_rows)
eda_audit_df = pd.DataFrame(eda_audit_rows)

image_stems = {p.stem for p in eda_image_paths}
label_stems = {p.stem for p in eda_label_paths}

images_without_label = sorted(image_stems - label_stems)
labels_without_image = sorted(label_stems - image_stems)

if len(eda_objects_df) > 0:
    object_count_by_class = eda_objects_df["class_id"].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index()
    image_count_by_class = (
        eda_objects_df.groupby("class_id")["image_id"]
        .nunique()
        .reindex(range(NUM_CLASSES), fill_value=0)
        .sort_index()
    )
else:
    object_count_by_class = pd.Series(0, index=range(NUM_CLASSES), dtype=int)
    image_count_by_class = pd.Series(0, index=range(NUM_CLASSES), dtype=int)

eda_class_stats = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "object_count": object_count_by_class.values,
    "image_count": image_count_by_class.values,
})

eda_class_stats["object_share"] = eda_class_stats["object_count"] / max(1, int(eda_class_stats["object_count"].sum()))
eda_class_stats["object_share_percent"] = eda_class_stats["object_share"] * 100

if len(eda_images_df) > 0:
    total_objects = int(eda_images_df["n_objects"].sum())
    images_with_objects = int((eda_images_df["n_objects"] > 0).sum())
    images_without_objects = int((eda_images_df["n_objects"] == 0).sum())
    mean_objects = float(eda_images_df["n_objects"].mean())
    median_objects = float(eda_images_df["n_objects"].median())
    max_objects = int(eda_images_df["n_objects"].max())
else:
    total_objects = 0
    images_with_objects = 0
    images_without_objects = 0
    mean_objects = 0.0
    median_objects = 0.0
    max_objects = 0

eda_overview = pd.DataFrame({
    "metric": [
        "train images",
        "train label files",
        "test images",
        "total valid objects",
        "images with >= 1 object",
        "images with 0 objects",
        "images without label file",
        "label files without image",
        "mean objects per image",
        "median objects per image",
        "max objects on one image",
        "malformed label lines",
        "invalid class lines",
        "invalid bbox lines",
        "unreadable images",
    ],
    "value": [
        len(eda_image_paths),
        len(eda_label_paths),
        len(eda_test_image_paths),
        total_objects,
        images_with_objects,
        images_without_objects,
        len(images_without_label),
        len(labels_without_image),
        round(mean_objects, 3),
        round(median_objects, 3),
        max_objects,
        int(eda_audit_df["malformed_lines"].sum()) if len(eda_audit_df) else 0,
        int(eda_audit_df["invalid_class_lines"].sum()) if len(eda_audit_df) else 0,
        int(eda_audit_df["invalid_bbox_lines"].sum()) if len(eda_audit_df) else 0,
        int((eda_images_df["image_read_ok"] == False).sum()) if len(eda_images_df) else 0,
    ],
})

print("\n==================== DATASET OVERVIEW ====================")
display(eda_overview)

print("\n==================== IMAGE SIZE DISTRIBUTION ====================")
if len(eda_images_df) > 0:
    display(
        eda_images_df["resolution"]
        .value_counts()
        .reset_index()
        .rename(columns={"index": "resolution", "resolution": "image_count"})
        .head(15)
    )

print("\n==================== OBJECTS PER IMAGE STATS ====================")
if len(eda_images_df) > 0:
    display(eda_images_df["n_objects"].describe().to_frame().T)
    display(
        eda_images_df[["image_id", "file_name", "width", "height", "n_objects"]]
        .sort_values("n_objects", ascending=False)
        .head(15)
    )

print("\n==================== CLASS DISTRIBUTION ====================")
display(eda_class_stats.sort_values("object_count", ascending=False).head(20))

zero_object_classes = eda_class_stats.loc[eda_class_stats["object_count"] == 0, "class_id"].tolist()
print("Classes without objects:", zero_object_classes if len(zero_object_classes) > 0 else "нет")

print("\n==================== BBOX STATS ====================")
if len(eda_objects_df) > 0:
    bbox_cols = [
        "bbox_width_norm",
        "bbox_height_norm",
        "bbox_area_norm",
        "bbox_aspect_ratio",
        "bbox_width_px",
        "bbox_height_px",
        "bbox_area_px",
    ]
    display(eda_objects_df[bbox_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)
else:
    print("В разметке не найдено валидных объектов.")

if len(images_without_label) > 0:
    print("\nПервые изображения без label-файла:")
    print(images_without_label[:20])

if len(labels_without_image) > 0:
    print("\nПервые label-файлы без изображения:")
    print(labels_without_image[:20])


# ============================================================
# Visualizations
# ============================================================

plt.figure(figsize=(18, 5))
plt.bar(eda_class_stats["class_id"].astype(str), eda_class_stats["object_count"])
plt.title("Распределение количества объектов по классам")
plt.xlabel("class_id")
plt.ylabel("Количество объектов")
plt.xticks(rotation=90)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
if len(eda_images_df) > 0:
    max_objects_for_bins = int(eda_images_df["n_objects"].max())
    bins = np.arange(0, max_objects_for_bins + 2) - 0.5
    plt.hist(eda_images_df["n_objects"], bins=bins)
plt.title("Распределение количества объектов на изображение")
plt.xlabel("Количество объектов на изображении")
plt.ylabel("Количество изображений")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 7))
top_classes = eda_class_stats.sort_values("object_count", ascending=False).head(20).sort_values("object_count")
plt.barh(top_classes["class_id"].astype(str), top_classes["object_count"])
plt.title("Топ-20 классов по количеству объектов")
plt.xlabel("Количество объектов")
plt.ylabel("class_id")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

if len(eda_objects_df) > 0:
    plt.figure(figsize=(8, 5))
    plt.hist(eda_objects_df["bbox_area_norm"], bins=50)
    plt.title("Распределение нормализованной площади bbox")
    plt.xlabel("bbox_width_norm × bbox_height_norm")
    plt.ylabel("Количество объектов")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 6))
    plt.scatter(
        eda_objects_df["bbox_width_norm"],
        eda_objects_df["bbox_height_norm"],
        s=10,
        alpha=0.25,
    )
    plt.title("Размеры bbox: ширина vs высота")
    plt.xlabel("bbox_width_norm")
    plt.ylabel("bbox_height_norm")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 6))
    plt.hist2d(
        eda_objects_df["x_center"],
        eda_objects_df["y_center"],
        bins=30,
        range=[[0, 1], [0, 1]],
    )
    plt.title("Где чаще находятся центры объектов")
    plt.xlabel("x_center")
    plt.ylabel("y_center")
    plt.colorbar(label="Количество объектов")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

if len(eda_images_df) > 0:
    top_resolutions = eda_images_df["resolution"].value_counts().head(10).sort_values()
    plt.figure(figsize=(10, 5))
    plt.barh(top_resolutions.index.astype(str), top_resolutions.values)
    plt.title("Топ разрешений изображений")
    plt.xlabel("Количество изображений")
    plt.ylabel("resolution")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


# ============================================================
# Optional visual sanity check: examples with many objects
# ============================================================

def eda_draw_yolo_boxes(image_rgb, bboxes, labels, thickness=2):
    """Рисует YOLO bbox поверх RGB-изображения только для EDA-визуализации."""
    image_vis = image_rgb.copy()
    h, w = image_vis.shape[:2]

    for bbox, label in zip(bboxes, labels):
        xc, yc, bw, bh = bbox
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        x1 = max(0, min(w - 1, x1))
        y1 = max(0, min(h - 1, y1))
        x2 = max(0, min(w - 1, x2))
        y2 = max(0, min(h - 1, y2))

        color = (
            int((37 * int(label) + 40) % 255),
            int((17 * int(label) + 120) % 255),
            int((29 * int(label) + 200) % 255),
        )
        cv2.rectangle(image_vis, (x1, y1), (x2, y2), color, thickness)
        cv2.putText(
            image_vis,
            str(int(label)),
            (x1, max(0, y1 - 4)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            color,
            1,
            cv2.LINE_AA,
        )

    return image_vis


def show_eda_images_with_most_objects(n=4):
    if len(eda_images_df) == 0:
        print("Нет изображений для визуализации.")
        return

    sample_df = eda_images_df.sort_values("n_objects", ascending=False).head(n)

    plt.figure(figsize=(6 * len(sample_df), 6))

    for idx, row in enumerate(sample_df.itertuples(index=False), start=1):
        image_path = Path(row.image_path)
        label_path = EDA_LABELS_DIR / f"{image_path.stem}.txt"

        image_rgb = read_image_rgb(image_path)
        bboxes, labels, _ = parse_yolo_label_for_eda(label_path)
        image_vis = eda_draw_yolo_boxes(image_rgb, bboxes, labels)

        plt.subplot(1, len(sample_df), idx)
        plt.imshow(image_vis)
        plt.title(f"{row.file_name}\nobjects: {row.n_objects}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


show_eda_images_with_most_objects(n=4)



## 5. Поиск датасета и разбиение train/valid

Здесь ноутбук использует уже найденные Colab-пути к изображениям и YOLO-разметке.

Дальше выполняется детерминированное разбиение на train и valid. Это нужно, чтобы:

- валидация была отделена от обучения;
- при повторном запуске получался тот же split;
- результаты экспериментов можно было сравнивать между собой честно.


In [ ]:
# ============================================================
# 5. Dataset resolver and deterministic train/valid split
# ============================================================

def find_yolo_image_label_dirs(search_root=None):
    search_root = Path(DATASET_SEARCH_ROOT if search_root is None else search_root)
    candidates = []

    for images_dir in search_root.rglob("images"):
        if not images_dir.is_dir():
            continue

        labels_dir = images_dir.parent / "labels"
        if not labels_dir.exists():
            continue

        image_count = len(collect_images(images_dir))
        if image_count == 0:
            continue

        path_text = str(images_dir).lower().replace("\\", "/")
        priority = 0

        if "/train/" in path_text or path_text.endswith("/train/images"):
            priority += 100
        if "valid" in path_text or "val" in path_text or "test" in path_text:
            priority -= 50

        candidates.append((priority, image_count, images_dir, labels_dir))

    if len(candidates) == 0:
        return None, None

    candidates = sorted(candidates, key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0][2], candidates[0][3]


def resolve_dataset_dirs():
    if MANUAL_TRAIN_IMAGES_DIR is not None and MANUAL_TRAIN_LABELS_DIR is not None:
        return Path(MANUAL_TRAIN_IMAGES_DIR), Path(MANUAL_TRAIN_LABELS_DIR)

    images_dir, labels_dir = find_yolo_image_label_dirs(DATASET_SEARCH_ROOT)
    if images_dir is None or labels_dir is None:
        raise FileNotFoundError(
            "Не удалось автоматически найти YOLO-структуру images/labels. "
            "Ошибочка, нужно прописать MANUAL_TRAIN_IMAGES_DIR и MANUAL_TRAIN_LABELS_DIR руками!!!"
        )

    return images_dir, labels_dir


def get_label_path_for_image(image_path):
    image_path = Path(image_path)
    return TRAIN_LABELS_DIR / f"{image_path.stem}.txt"


def split_train_valid(image_paths, valid_fraction=0.15):
    train_paths, valid_paths = [], []

    for path in image_paths:
        value = stable_int_hash(Path(path).stem) / float(2**32 - 1)
        if value < valid_fraction:
            valid_paths.append(path)
        else:
            train_paths.append(path)

    if len(valid_paths) == 0 and len(image_paths) > 1:
        n_valid = max(1, int(len(image_paths) * valid_fraction))
        valid_paths = image_paths[:n_valid]
        train_paths = image_paths[n_valid:]

    return train_paths, valid_paths


TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR = resolve_dataset_dirs()
all_image_paths = collect_images(TRAIN_IMAGES_DIR)

if MAX_IMAGES_DEBUG is not None:
    all_image_paths = all_image_paths[:MAX_IMAGES_DEBUG]

train_image_paths, valid_image_paths = split_train_valid(
    all_image_paths,
    valid_fraction=VALID_FRACTION,
)

print("TRAIN_IMAGES_DIR:", TRAIN_IMAGES_DIR)
print("TRAIN_LABELS_DIR:", TRAIN_LABELS_DIR)
print("Total .jpg train images found:", len(all_image_paths))
print("Train images:", len(train_image_paths))
print("Valid images:", len(valid_image_paths))

assert len(all_image_paths) > 0, "Не найдено ни одной .jpg картинки в TRAIN_IMAGES_DIR"
assert len(train_image_paths) > 0, "После split train оказался пустым"
assert len(valid_image_paths) > 0, "После split valid оказался пустым"


## 6. Совместимость с Albumentations

В разных версиях `albumentations` некоторые параметры могут называться или поддерживаться немного по-разному. Поэтому здесь собраны обёртки, которые делают код устойчивее к версии библиотеки в Colab.

Главная задача блока — корректно настроить `BboxParams`, чтобы аугментации не ломали YOLO-разметку и сохраняли связь между изображением, bounding boxes и классами.


In [ ]:
# ============================================================
# 6. Albumentations compatibility helpers
# ============================================================

def make_bbox_params():
    params = inspect.signature(A.BboxParams).parameters
    kwargs = dict(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.0,
        min_area=0,
    )

    if "clip" in params:
        kwargs["clip"] = True

    return A.BboxParams(**kwargs)


def make_albu_gauss_noise(
    std_range=(0.06, 0.18),
    var_limit=(70.0, 420.0),
    per_channel=True,
    p=1.0,
):
    params = inspect.signature(A.GaussNoise).parameters

    if "var_limit" in params:
        kwargs = dict(var_limit=var_limit, mean=0, p=p)
        if "per_channel" in params:
            kwargs["per_channel"] = per_channel
        return A.GaussNoise(**kwargs)

    kwargs = dict(std_range=std_range, mean_range=(0.0, 0.0), p=p)
    if "per_channel" in params:
        kwargs["per_channel"] = per_channel
    return A.GaussNoise(**kwargs)


def resolve_severity(severity):
    if severity in ["medium", "hard", "extreme"]:
        return severity

    return str(
        np.random.choice(
            ["medium", "hard", "extreme"],
            p=[0.18, 0.37, 0.45],
        )
    )


## 7. Dense dirty augmentation: имитация грязи и артефактов

Это основной кастомный блок аугментаций. Он создаёт плотные полупрозрачные пятна, разводы и неоднородные артефакты, похожие на загрязнение линзы или плохое качество съёмки.

Важно: эта аугментация **не меняет геометрию объектов**, поэтому bounding boxes остаются валидными. Меняется только внешний вид изображения!!!

Идея этого блока — сделать train ближе к test, где изображения выглядят испорченными: с грязью, мутностью, пятнами, шумом и потерей деталей, я не знаю, что это за аугументация, но визуально так и выглядит!.


In [ ]:
# ============================================================
# 7. Dense dirty augmentation
# ============================================================

DENSE_DIRTY_OVERLAY_ALPHA_RANGE = (0.62, 0.94)


def normalize01(x):
    x = x.astype(np.float32)
    x = x - x.min()
    denom = x.max() - x.min()
    if denom < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return x / denom


def smooth_cyclic_values(values, n_iter=3):
    values = values.astype(np.float32)
    for _ in range(n_iter):
        values = (
            np.roll(values, -2)
            + 2 * np.roll(values, -1)
            + 3 * values
            + 2 * np.roll(values, 1)
            + np.roll(values, 2)
        ) / 9.0
    return values


def make_fractal_noise(h, w, rng, scales=(8, 16, 32, 64), weights=(1.0, 0.8, 0.5, 0.25)):
    noise = np.zeros((h, w), dtype=np.float32)

    for scale, weight in zip(scales, weights):
        sh = max(2, h // scale)
        sw = max(2, w // scale)

        small = rng.rand(sh, sw).astype(np.float32)
        up = cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC)

        k = max(3, min(h, w) // max(8, scale // 2))
        if k % 2 == 0:
            k += 1

        up = cv2.GaussianBlur(up, (k, k), 0)
        noise += weight * up

    return normalize01(noise)


def make_dirty_cloud_alpha(
    h,
    w,
    rng,
    threshold_range=(0.18, 0.30),
    softness_range=(12.0, 24.0),
    blur_range=(9, 19),
    density_scale=1.70,
):
    base1 = make_fractal_noise(h, w, rng, scales=(6, 12, 24, 48), weights=(1.0, 0.85, 0.55, 0.30))
    base2 = make_fractal_noise(h, w, rng, scales=(10, 20, 40, 80), weights=(1.0, 0.75, 0.45, 0.22))
    base3 = make_fractal_noise(h, w, rng, scales=(8, 16, 32), weights=(1.0, 0.65, 0.35))

    mix = normalize01(0.45 * base1 + 0.35 * base2 + 0.20 * base3)

    threshold = rng.uniform(*threshold_range)
    threshold = np.clip(threshold / density_scale, 0.10, 0.65)
    softness = rng.uniform(*softness_range)

    alpha = 1.0 / (1.0 + np.exp(-softness * (mix - threshold)))
    alpha = normalize01(alpha)

    k = rng.randint(blur_range[0], blur_range[1] + 1)
    if k % 2 == 0:
        k += 1
    alpha = cv2.GaussianBlur(alpha, (k, k), 0)

    texture1 = make_fractal_noise(h, w, rng, scales=(10, 20, 40), weights=(1.0, 0.6, 0.3))
    texture2 = make_fractal_noise(h, w, rng, scales=(14, 28, 56), weights=(1.0, 0.5, 0.25))

    alpha = alpha * (0.75 + 0.85 * texture1) + 0.18 * texture2
    alpha = normalize01(alpha)

    alpha = np.power(alpha, rng.uniform(0.75, 0.95))
    return normalize01(alpha)


def make_irregular_splat_patch(
    rng,
    size_range=(7, 50),
    n_vertices_range=(18, 34),
    roughness_range=(0.18, 0.48),
    blur_sigma_range=(0.60, 1.50),
    alpha_range=(0.50, 1.00),
    supersample=4,
):
    size = rng.randint(size_range[0], size_range[1] + 1)
    if size % 2 == 0:
        size += 1

    ss = supersample
    big_size = size * ss
    canvas = np.zeros((big_size, big_size), dtype=np.float32)

    cx = big_size * rng.uniform(0.43, 0.57)
    cy = big_size * rng.uniform(0.43, 0.57)
    base_radius = big_size * rng.uniform(0.18, 0.35)

    n_vertices = rng.randint(n_vertices_range[0], n_vertices_range[1] + 1)
    angles = np.linspace(0, 2 * np.pi, n_vertices, endpoint=False)

    noise = rng.normal(0, 1, size=n_vertices).astype(np.float32)
    noise = smooth_cyclic_values(noise, n_iter=rng.randint(2, 5))
    noise = normalize01(noise) * 2.0 - 1.0

    roughness = rng.uniform(*roughness_range)
    radii = base_radius * (1.0 + roughness * noise)
    radii = np.clip(radii, base_radius * 0.45, base_radius * 1.65)

    points = []
    for angle, radius in zip(angles, radii):
        x = cx + np.cos(angle) * radius
        y = cy + np.sin(angle) * radius
        points.append([int(round(x)), int(round(y))])

    cv2.fillPoly(canvas, [np.array(points, dtype=np.int32)], 1.0)

    # Округлые выступы: убираем квадратность и делаем форму похожей на кляксу.
    for _ in range(rng.randint(1, 5)):
        angle = rng.uniform(0, 2 * np.pi)
        dist = base_radius * rng.uniform(0.35, 0.95)
        lx = int(round(cx + np.cos(angle) * dist))
        ly = int(round(cy + np.sin(angle) * dist))
        lr = max(2 * ss, int(round(base_radius * rng.uniform(0.20, 0.48))))
        cv2.circle(canvas, (lx, ly), lr, rng.uniform(0.55, 1.0), -1)

    # Мягкие вырезы по краям, чтобы пятна не были идеальными окружностями.
    if rng.rand() < 0.45:
        for _ in range(rng.randint(1, 3)):
            angle = rng.uniform(0, 2 * np.pi)
            dist = base_radius * rng.uniform(0.70, 1.25)
            hx = int(round(cx + np.cos(angle) * dist))
            hy = int(round(cy + np.sin(angle) * dist))
            hr = max(2 * ss, int(round(base_radius * rng.uniform(0.16, 0.35))))

            hole = np.zeros_like(canvas)
            cv2.circle(hole, (hx, hy), hr, 1.0, -1)
            canvas = canvas * (1.0 - 0.65 * hole)

    sigma = rng.uniform(*blur_sigma_range) * ss
    canvas = cv2.GaussianBlur(canvas, (0, 0), sigmaX=sigma, sigmaY=sigma)

    inner_noise_small = rng.rand(max(3, size // 4), max(3, size // 4)).astype(np.float32)
    inner_noise = cv2.resize(inner_noise_small, (big_size, big_size), interpolation=cv2.INTER_CUBIC)
    inner_noise = cv2.GaussianBlur(inner_noise, (0, 0), sigmaX=rng.uniform(0.8, 1.8) * ss)
    inner_noise = normalize01(inner_noise)

    canvas = canvas * (0.70 + 0.45 * inner_noise)
    patch = cv2.resize(canvas, (size, size), interpolation=cv2.INTER_AREA)
    patch = normalize01(patch)
    patch *= rng.uniform(*alpha_range)

    return patch.astype(np.float32)


def paste_max_patch(canvas, patch, center_x, center_y):
    h, w = canvas.shape
    ph, pw = patch.shape

    x1 = center_x - pw // 2
    y1 = center_y - ph // 2
    x2 = x1 + pw
    y2 = y1 + ph

    cx1 = max(0, x1)
    cy1 = max(0, y1)
    cx2 = min(w, x2)
    cy2 = min(h, y2)

    if cx1 >= cx2 or cy1 >= cy2:
        return canvas

    px1 = cx1 - x1
    py1 = cy1 - y1
    px2 = px1 + (cx2 - cx1)
    py2 = py1 + (cy2 - cy1)

    canvas[cy1:cy2, cx1:cx2] = np.maximum(
        canvas[cy1:cy2, cx1:cx2],
        patch[py1:py2, px1:px2],
    )

    return canvas


def make_dark_inclusions(
    h,
    w,
    rng,
    cloud_alpha,
    n_spots_range=(850, 1600),
    size_range=(7, 50),
    spot_alpha_range=(0.18, 0.46),
    cloud_threshold=0.045,
):
    inclusion = np.zeros((h, w), dtype=np.float32)
    ys, xs = np.where(cloud_alpha > cloud_threshold)

    if len(xs) == 0:
        return inclusion

    n_spots = rng.randint(n_spots_range[0], n_spots_range[1] + 1)

    for _ in range(n_spots):
        idx = rng.randint(0, len(xs))
        x = int(xs[idx])
        y = int(ys[idx])

        patch = make_irregular_splat_patch(
            rng,
            size_range=size_range,
            n_vertices_range=(18, 34),
            roughness_range=(0.18, 0.48),
            blur_sigma_range=(0.60, 1.50),
            alpha_range=(0.50, 1.00),
            supersample=4,
        )

        patch *= rng.uniform(*spot_alpha_range)
        inclusion = paste_max_patch(inclusion, patch, x, y)

    inclusion *= 0.45 + 1.25 * cloud_alpha
    inclusion = np.clip(inclusion, 0, 1)

    if rng.rand() < 0.75:
        inclusion = cv2.GaussianBlur(inclusion, (3, 3), 0)

    return np.clip(inclusion, 0, 1)


def make_micro_dirty_texture(
    h,
    w,
    rng,
    cloud_alpha,
    density_range=(0.018, 0.042),
    alpha_range=(0.035, 0.11),
    radius_range=(1, 3),
    blur_prob=0.70,
):
    texture = np.zeros((h, w), dtype=np.float32)
    ys, xs = np.where(cloud_alpha > 0.08)

    if len(xs) == 0:
        return texture

    n = int(h * w * rng.uniform(*density_range))
    if n <= 0:
        return texture

    chosen = rng.choice(len(xs), size=min(n, len(xs)), replace=False)

    for idx in chosen:
        x = int(xs[idx])
        y = int(ys[idx])
        r = rng.randint(radius_range[0], radius_range[1] + 1)
        val = rng.uniform(*alpha_range)

        if rng.rand() < 0.75:
            cv2.circle(texture, (x, y), r, val, -1)
        else:
            rx = r
            ry = max(1, int(round(r * rng.uniform(0.7, 1.4))))
            angle = rng.uniform(0, 180)
            cv2.ellipse(texture, (x, y), (rx, ry), angle, 0, 360, val, -1)

    if rng.rand() < blur_prob:
        texture = cv2.GaussianBlur(texture, (3, 3), 0)

    texture *= 0.35 + 1.20 * cloud_alpha
    return np.clip(texture, 0, 1)


def apply_dirty_cloud_overlay(image, rng, cloud_alpha, dark_inclusions, micro_texture):
    img = image.astype(np.float32).copy()

    tint = np.array(
        [
            rng.uniform(75, 130),
            rng.uniform(65, 115),
            rng.uniform(50, 95),
        ],
        dtype=np.float32,
    )

    overlay_alpha = rng.uniform(*DENSE_DIRTY_OVERLAY_ALPHA_RANGE)
    cloud_main = np.power(cloud_alpha, rng.uniform(0.70, 0.95))
    a = np.clip(cloud_main * overlay_alpha, 0, 0.97)[..., None]

    overlay = np.ones_like(img, dtype=np.float32) * tint.reshape(1, 1, 3)
    mixed = img * (1.0 - a) + overlay * a

    dark_map = np.clip(1.00 * dark_inclusions + 0.55 * micro_texture, 0, 0.98)
    dark_a = dark_map[..., None]

    dark_strength = rng.uniform(0.03, 0.22)
    mixed = mixed * (1.0 - dark_a) + (mixed * dark_strength) * dark_a

    blurred = cv2.GaussianBlur(mixed, (7, 7), 0)
    blur_a = np.clip(cloud_alpha * rng.uniform(0.18, 0.34), 0, 0.50)[..., None]
    mixed = mixed * (1.0 - blur_a) + blurred * blur_a

    return as_uint8(mixed)


def augment_dirty_clouds_dense_only(image, seed=None):
    rng = np.random.RandomState(seed if seed is not None else np.random.randint(0, 10_000_000))
    image = as_uint8(image)
    h, w = image.shape[:2]

    cloud_alpha = make_dirty_cloud_alpha(
        h,
        w,
        rng,
        threshold_range=(0.18, 0.30),
        softness_range=(12.0, 24.0),
        blur_range=(9, 19),
        density_scale=1.70,
    )

    dark_inclusions = make_dark_inclusions(
        h,
        w,
        rng,
        cloud_alpha,
        n_spots_range=(850, 1600),
        size_range=(7, 50),
        spot_alpha_range=(0.18, 0.46),
        cloud_threshold=0.045,
    )

    micro_texture = make_micro_dirty_texture(
        h,
        w,
        rng,
        cloud_alpha,
        density_range=(0.018, 0.042),
        alpha_range=(0.035, 0.11),
        radius_range=(1, 3),
    )

    return apply_dirty_cloud_overlay(
        image,
        rng,
        cloud_alpha=cloud_alpha,
        dark_inclusions=dark_inclusions,
        micro_texture=micro_texture,
    )


class DenseDirtyFromNotebook(A.ImageOnlyTransform):
    def __init__(self, p=0.55):
        super().__init__(p=p)

    def apply(self, img, **params):
        seed = int(np.random.randint(0, 10_000_000))
        return augment_dirty_clouds_dense_only(img, seed=seed)

    def get_transform_init_args_names(self):
        return ()


## 8. Финальное ухудшение качества после dirty-аугментации

После наложения грязи изображение дополнительно портится: применяется blur, сжатие, шум и другие деградации качества.

Порядок здесь принципиален:

1. сначала добавляются dirty-артефакты;
2. затем всё изображение вместе с артефактами размывается или сжимается.

Так визуально получается эффект, будто грязь действительно была на линзе/изображении до финального ухудшения качества, а не просто наложена поверх уже готовой картинки.


In [ ]:
# ============================================================
# 8. Final post-dirty quality degradation with stronger MedianBlur
# ============================================================

class PostDirtyQualityDegradation(A.ImageOnlyTransform):
    def __init__(self, severity="random", p=1.0):
        super().__init__(p=p)
        self.severity = severity

    def _resolve_severity(self):
        if self.severity in ["medium", "hard", "extreme"]:
            return self.severity

        return str(
            np.random.choice(
                ["medium", "hard", "extreme"],
                p=[0.20, 0.36, 0.44],
            )
        )

    def _params_by_severity(self, severity):
        if severity == "medium":
            return dict(
                scale_range=(0.55, 0.85),
                jpeg_quality_range=(35, 68),
                gaussian_kernels=[3, 5],
                median_kernels=[3, 5],
                gaussian_sigma_range=(0.35, 1.00),
                median_prob=0.48,
            )

        if severity == "hard":
            return dict(
                scale_range=(0.44, 0.80),
                jpeg_quality_range=(22, 56),
                gaussian_kernels=[3, 5, 7],
                median_kernels=[5, 7, 9],
                gaussian_sigma_range=(0.40, 1.05),
                median_prob=0.58,
            )

        return dict(
            scale_range=(0.25, 0.55),
            jpeg_quality_range=(8, 34),
            gaussian_kernels=[7, 9, 11],
            median_kernels=[7, 9, 11, 13],
            gaussian_sigma_range=(0.80, 1.70),
            median_prob=0.66,
        )

    def apply(self, img, **params):
        img = as_uint8(img).copy()
        h, w = img.shape[:2]

        severity = self._resolve_severity()
        p = self._params_by_severity(severity)

        # 1) Сначала снижаем разрешение.
        scale = float(np.random.uniform(*p["scale_range"]))
        new_w = max(16, int(w * scale))
        new_h = max(16, int(h * scale))

        small = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        upscale_interpolation = int(
            np.random.choice(
                [
                    cv2.INTER_LINEAR,
                    cv2.INTER_CUBIC,
                    cv2.INTER_NEAREST,
                ]
            )
        )
        img = cv2.resize(small, (w, h), interpolation=upscale_interpolation)

        # 2) Потом JPEG-сжатие.
        jpeg_quality = int(
            np.random.randint(
                p["jpeg_quality_range"][0],
                p["jpeg_quality_range"][1] + 1,
            )
        )

        success, encimg = cv2.imencode(
            ".jpg",
            cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
            [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality],
        )

        if success:
            dec = cv2.imdecode(encimg, cv2.IMREAD_COLOR)
            img = cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)

        # 3) Blur строго после снижения качества.
        median_prob = float(p["median_prob"])
        blur_type = str(
            np.random.choice(
                ["gaussian", "median"],
                p=[1.0 - median_prob, median_prob],
            )
        )

        if blur_type == "gaussian":
            k = int(np.random.choice(p["gaussian_kernels"]))
            sigma = float(np.random.uniform(*p["gaussian_sigma_range"]))
            img = cv2.GaussianBlur(img, (k, k), sigmaX=sigma, sigmaY=sigma)
        else:
            k = int(np.random.choice(p["median_kernels"]))
            if k % 2 == 0:
                k += 1
            img = cv2.medianBlur(img, k)

        return as_uint8(img)

    def get_transform_init_args_names(self):
        return ("severity",)


## 9. Дополнительные негеометрические аугментации

В этом блоке добавлены дополнительные искажения, которые не двигают объекты и не требуют пересчёта координат рамок:

- перевод в серый формат с сохранением трёх каналов;
- шумы;
- изменение яркости/контраста;
- blur-эффекты;
- ухудшение резкости и качества.


Согласен, что выглядит странно, но делал это немного поздно.

Но визуально те изображения в тесте, которые идут с этой непонятной мутью, все с ухудшенным качеством и каким-то Blur-ом, поэтому так неоптимизировано и глупо это реализовал.

Честно, без понятия, что это за аугументация

In [ ]:
# ============================================================
# 9. Additional non-geometric augmentations
# ============================================================

class ToGray3(A.ImageOnlyTransform):
    def __init__(self, p=0.2):
        super().__init__(p=p)

    def apply(self, img, **params):
        img = as_uint8(img)
        if img.ndim == 2:
            gray = img
        else:
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

    def get_transform_init_args_names(self):
        return ()


class StrongRGBNoise(A.ImageOnlyTransform):
    def __init__(self, severity="random", p=0.20):
        super().__init__(p=p)
        self.severity = severity

    def apply(self, img, **params):
        severity = resolve_severity(self.severity)
        img = as_uint8(img).astype(np.float32)

        if severity == "medium":
            sigma = np.random.uniform(12, 28)
            gray_sigma_part = np.random.uniform(0.15, 0.28)
            mult_sigma = np.random.uniform(0.03, 0.08)
        elif severity == "hard":
            sigma = np.random.uniform(28, 50)
            gray_sigma_part = np.random.uniform(0.18, 0.35)
            mult_sigma = np.random.uniform(0.07, 0.14)
        else:
            sigma = np.random.uniform(50, 72)
            gray_sigma_part = np.random.uniform(0.22, 0.42)
            mult_sigma = np.random.uniform(0.10, 0.20)

        img += np.random.normal(0, sigma, size=img.shape).astype(np.float32)

        if np.random.rand() < 0.42:
            gray_noise = np.random.normal(0, sigma * gray_sigma_part, size=img.shape[:2]).astype(np.float32)
            img += gray_noise[:, :, None]

        if np.random.rand() < 0.45:
            mult = np.random.normal(1.0, mult_sigma, size=img.shape).astype(np.float32)
            img *= mult

        return np.clip(img, 0, 255).astype(np.uint8)

    def get_transform_init_args_names(self):
        return ("severity",)


class SquareBlackDustDots(A.ImageOnlyTransform):
    def __init__(self, severity="random", p=0.78):
        super().__init__(p=p)
        self.severity = severity

    def apply(self, img, **params):
        severity = resolve_severity(self.severity)
        out = as_uint8(img).copy()
        h, w = out.shape[:2]
        area = h * w

        if severity == "medium":
            coverage_range = (0.006, 0.016)
            sizes = [1, 1, 2, 2, 3]
        elif severity == "hard":
            coverage_range = (0.013, 0.032)
            sizes = [1, 1, 2, 2, 3, 3, 4]
        else:
            coverage_range = (0.022, 0.055)
            sizes = [2, 2, 3, 3, 4]

        size = int(np.random.choice(sizes))
        coverage = float(np.random.uniform(*coverage_range))
        n = min(int((area * coverage) / max(1, size * size)), 9000)

        value = int(np.random.randint(0, 20))
        alpha = float(np.random.uniform(0.86, 1.00))

        for _ in range(n):
            x = int(np.random.randint(0, max(1, w - size + 1)))
            y = int(np.random.randint(0, max(1, h - size + 1)))

            patch = out[y:y + size, x:x + size].astype(np.float32)
            color = np.full_like(patch, value, dtype=np.float32)
            out[y:y + size, x:x + size] = np.clip(patch * (1.0 - alpha) + color * alpha, 0, 255).astype(np.uint8)

        return out

    def get_transform_init_args_names(self):
        return ("severity",)


class SquareWhiteDustDots(A.ImageOnlyTransform):
    def __init__(self, severity="random", p=0.46):
        super().__init__(p=p)
        self.severity = severity

    def apply(self, img, **params):
        severity = resolve_severity(self.severity)
        out = as_uint8(img).copy()
        h, w = out.shape[:2]
        area = h * w

        if severity == "medium":
            coverage_range = (0.0018, 0.0050)
            sizes = [1, 1, 2, 2]
        elif severity == "hard":
            coverage_range = (0.0030, 0.0085)
            sizes = [1, 2, 2, 3]
        else:
            coverage_range = (0.0050, 0.0140)
            sizes = [2, 2, 3, 3, 4]

        size = int(np.random.choice(sizes))
        coverage = float(np.random.uniform(*coverage_range))
        n = min(int((area * coverage) / max(1, size * size)), 6500)

        value = int(np.random.randint(230, 256))
        alpha = float(np.random.uniform(0.62, 0.92)) if size >= 3 else float(np.random.uniform(0.78, 1.00))

        for _ in range(n):
            x = int(np.random.randint(0, max(1, w - size + 1)))
            y = int(np.random.randint(0, max(1, h - size + 1)))

            patch = out[y:y + size, x:x + size].astype(np.float32)
            color = np.full_like(patch, value, dtype=np.float32)
            out[y:y + size, x:x + size] = np.clip(patch * (1.0 - alpha) + color * alpha, 0, 255).astype(np.uint8)

        return out

    def get_transform_init_args_names(self):
        return ("severity",)


class RandomThinStripes(A.ImageOnlyTransform):
    def __init__(self, severity="random", p=0.24):
        super().__init__(p=p)
        self.severity = severity

    def _params_by_severity(self, severity):
        if severity == "medium":
            return dict(n_range=(1, 3), thickness_choices=[1, 1, 2], alpha_range=(0.14, 0.32))
        if severity == "hard":
            return dict(n_range=(2, 5), thickness_choices=[1, 1, 2, 2, 3], alpha_range=(0.18, 0.42))
        return dict(n_range=(3, 7), thickness_choices=[1, 2, 2, 3], alpha_range=(0.24, 0.52))

    def apply(self, img, **params):
        severity = resolve_severity(self.severity)
        cfg = self._params_by_severity(severity)
        out = as_uint8(img).copy()
        h, w = out.shape[:2]

        orientation = str(np.random.choice(["horizontal", "vertical"]))
        n_lines = int(np.random.randint(cfg["n_range"][0], cfg["n_range"][1] + 1))

        for _ in range(n_lines):
            overlay = out.copy()
            value = int(np.random.randint(0, 35))
            alpha = float(np.random.uniform(*cfg["alpha_range"]))
            thickness = int(np.random.choice(cfg["thickness_choices"]))

            if orientation == "horizontal":
                y = int(np.random.randint(0, h))
                y1 = max(0, y - thickness)
                y2 = min(h - 1, y + thickness)
                cv2.rectangle(overlay, (0, y1), (w - 1, y2), (value, value, value), -1)
            else:
                x = int(np.random.randint(0, w))
                x1 = max(0, x - thickness)
                x2 = min(w - 1, x + thickness)
                cv2.rectangle(overlay, (x1, 0), (x2, h - 1), (value, value, value), -1)

            out = cv2.addWeighted(overlay, alpha, out, 1.0 - alpha, 0)

        return out

    def get_transform_init_args_names(self):
        return ("severity",)


## 10. Финальные пайплайны аугментаций

Здесь собираются итоговые `train_transform` и `valid_transform`.

Логика пайплайна такая:

1. применить dirty-аугментацию;
2. обязательно ухудшить качество изображения;
3. добавить дополнительные негеометрические искажения;
4. сохранить YOLO bounding boxes в корректном формате.

YOLO-внутренние аугментации дальше отключаются, потому что основная логика искажений уже контролируется вручную через этот блок.


In [ ]:
# ============================================================
# 10. Build final augmentation pipelines
# ============================================================

def build_train_transform(dense_dirty_p=0.55):
    return A.Compose(
        [
            # 1) Грязь первой.
            DenseDirtyFromNotebook(p=dense_dirty_p),

            # 2) Потом обязательное снижение качества и blur.
            PostDirtyQualityDegradation(severity="random", p=1.00),

            # 3) Дополнительные эффекты без геометрии.
            ToGray3(p=0.20),

            A.OneOf(
                [
                    make_albu_gauss_noise(
                        std_range=(0.035, 0.085),
                        var_limit=(35.0, 180.0),
                        per_channel=True,
                        p=0.62,
                    ),
                    StrongRGBNoise(severity="medium", p=0.25),
                    StrongRGBNoise(severity="hard", p=0.13),
                ],
                p=0.42,
            ),

            SquareBlackDustDots(severity="random", p=0.78),
            SquareWhiteDustDots(severity="random", p=0.24),
            RandomThinStripes(severity="random", p=0.30),

            A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.16, p=0.22),
            A.RandomGamma(gamma_limit=(82, 118), p=0.18),
        ],
        bbox_params=make_bbox_params(),
    )


def build_valid_transform_matched():
    return A.Compose(
        [
            DenseDirtyFromNotebook(p=0.46),
            PostDirtyQualityDegradation(severity="random", p=1.00),
            ToGray3(p=0.16),

            A.OneOf(
                [
                    make_albu_gauss_noise(
                        std_range=(0.030, 0.075),
                        var_limit=(30.0, 155.0),
                        per_channel=True,
                        p=0.68,
                    ),
                    StrongRGBNoise(severity="medium", p=0.24),
                    StrongRGBNoise(severity="hard", p=0.08),
                ],
                p=0.36,
            ),

            SquareBlackDustDots(severity="random", p=0.62),
            SquareWhiteDustDots(severity="random", p=0.20),
            RandomThinStripes(severity="random", p=0.18),
            A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.12, p=0.14),
        ],
        bbox_params=make_bbox_params(),
    )


def build_valid_transform_lighter():
    return A.Compose(
        [
            DenseDirtyFromNotebook(p=0.30),
            PostDirtyQualityDegradation(severity="medium", p=1.00),
            ToGray3(p=0.10),

            A.OneOf(
                [
                    make_albu_gauss_noise(
                        std_range=(0.020, 0.055),
                        var_limit=(18.0, 110.0),
                        per_channel=True,
                        p=0.76,
                    ),
                    StrongRGBNoise(severity="medium", p=0.24),
                ],
                p=0.26,
            ),

            SquareBlackDustDots(severity="medium", p=0.42),
            SquareWhiteDustDots(severity="medium", p=0.20),
            RandomThinStripes(severity="medium", p=0.10),
            A.RandomBrightnessContrast(brightness_limit=0.06, contrast_limit=0.10, p=0.10),
        ],
        bbox_params=make_bbox_params(),
    )


train_transform = build_train_transform(dense_dirty_p=0.55)
valid_transform_matched = build_valid_transform_matched()
valid_transform_lighter = build_valid_transform_lighter()
valid_transform = valid_transform_matched if USE_DIRTY_VALID else valid_transform_lighter

demo_transform = build_train_transform(dense_dirty_p=1.00)

print("Final train_transform created")
print("Final valid_transform created")
print("Demo transform uses the same final pipeline, but forces dense dirty p=1.00")


## 11. Визуальная проверка аугментаций

Перед обучением важно глазами проверить, что аугментации выглядят реалистично и не ломают разметку.


In [ ]:
# ============================================================
# 11. Visualize ONLY the final augmentation version
# ============================================================


def draw_yolo_boxes(image_rgb, bboxes, labels, thickness=2):
    image = image_rgb.copy()
    h, w = image.shape[:2]

    for bbox, label in zip(bboxes, labels):
        xc, yc, bw, bh = map(float, bbox)

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        x1 = max(0, min(w - 1, x1))
        y1 = max(0, min(h - 1, y1))
        x2 = max(0, min(w - 1, x2))
        y2 = max(0, min(h - 1, y2))

        cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), thickness)
        cv2.putText(
            image,
            str(int(label)),
            (x1, max(0, y1 - 4)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 0, 0),
            1,
            cv2.LINE_AA,
        )

    return image


def apply_demo_final_aug(image_rgb, bboxes, labels, seed=SEED):
    rng_state = np.random.get_state()
    py_state = random.getstate()

    np.random.seed(seed)
    random.seed(seed)

    result = demo_transform(
        image=image_rgb,
        bboxes=bboxes,
        class_labels=labels,
    )

    np.random.set_state(rng_state)
    random.setstate(py_state)

    return result["image"], list(result["bboxes"]), list(result["class_labels"])


def visualize_final_augmentation(n=8):
    rng = random.Random(SEED)
    sample_paths = rng.sample(train_image_paths, min(n, len(train_image_paths)))

    for idx, image_path in enumerate(sample_paths):
        image_path = Path(image_path)
        image_rgb = read_image_rgb(image_path)
        bboxes, labels = read_yolo_label(get_label_path_for_image(image_path))

        aug_image, aug_bboxes, aug_labels = apply_demo_final_aug(
            image_rgb=image_rgb,
            bboxes=bboxes,
            labels=labels,
            seed=SEED + idx * 1000 + 17,
        )

        original_vis = draw_yolo_boxes(image_rgb, bboxes, labels)
        final_vis = draw_yolo_boxes(aug_image, aug_bboxes, aug_labels)

        plt.figure(figsize=(14, 7))

        plt.subplot(1, 2, 1)
        plt.imshow(original_vis)
        plt.title(f"Original: {image_path.name}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(final_vis)
        plt.title("Final augmentation only")
        plt.axis("off")

        plt.tight_layout()
        plt.show()


visualize_final_augmentation(n=8)


## 12. Материализация offline-аугментированного YOLO-датасета

В этом блоке создаётся новый датасет в формате YOLO внутри `/content/working`.

Для каждого изображения генерируются аугментированные копии, после чего сохраняются:

- изображения в `images/train` и `images/valid`;
- соответствующие `.txt`-файлы с YOLO-разметкой в `labels/train` и `labels/valid`;
- `data.yaml`, который затем передаётся в `YOLO.train()`.


In [ ]:
# ============================================================
# 12. Materialize offline augmented YOLO dataset
# ============================================================

def reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def make_aug_output_name(idx, copy_idx, image_path, prefix="aug"):
    image_path = Path(image_path)
    return f"{idx:05d}_{prefix}{copy_idx}_{image_path.stem}.jpg"


def set_aug_seed_for_item(copy_idx, split_name, image_path):
    key = f"{split_name}_{copy_idx}_{Path(image_path).stem}"
    item_seed = stable_int_hash(key, seed=SEED)
    random.seed(item_seed)
    np.random.seed(item_seed)


def apply_transform_to_image_and_labels(image_path, transform, copy_idx, split_name):
    image_rgb = read_image_rgb(image_path)
    bboxes, labels = read_yolo_label(get_label_path_for_image(image_path))

    set_aug_seed_for_item(
        copy_idx=copy_idx,
        split_name=split_name,
        image_path=image_path,
    )

    result = transform(
        image=image_rgb,
        bboxes=bboxes,
        class_labels=labels,
    )

    return result["image"], list(result["bboxes"]), list(result["class_labels"])


def write_data_yaml_offline():
    data = {
        "path": str(OFFLINE_ROOT),
        "train": "images/train",
        "val": "images/valid",
        "nc": int(NUM_CLASSES),
        "names": [str(i) for i in range(NUM_CLASSES)],
    }

    with open(OFFLINE_DATA_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

    print("data.yaml:", OFFLINE_DATA_YAML)


def materialize_augmented_split(
    image_paths,
    out_images_dir,
    out_labels_dir,
    transform,
    split_name,
    copies_per_image,
    keep_original=False,
):
    out_images_dir = Path(out_images_dir)
    out_labels_dir = Path(out_labels_dir)

    total = len(image_paths) * copies_per_image + (len(image_paths) if keep_original else 0)
    pbar = tqdm(total=total, desc=f"Materialize {split_name}")

    written_images = 0

    for idx, image_path in enumerate(image_paths):
        image_path = Path(image_path)

        if keep_original:
            out_name = make_aug_output_name(idx, 0, image_path, prefix="orig")
            out_image_path = out_images_dir / out_name
            out_label_path = out_labels_dir / f"{Path(out_name).stem}.txt"

            image_rgb = read_image_rgb(image_path)
            bboxes, labels = read_yolo_label(get_label_path_for_image(image_path))

            save_image_rgb_jpg(out_image_path, image_rgb, quality=96)
            write_yolo_label(out_label_path, bboxes, labels)

            written_images += 1
            pbar.update(1)

        for copy_idx in range(copies_per_image):
            out_name = make_aug_output_name(idx, copy_idx, image_path, prefix="aug")
            out_image_path = out_images_dir / out_name
            out_label_path = out_labels_dir / f"{Path(out_name).stem}.txt"

            aug_image, aug_bboxes, aug_labels = apply_transform_to_image_and_labels(
                image_path=image_path,
                transform=transform,
                copy_idx=copy_idx,
                split_name=split_name,
            )

            save_image_rgb_jpg(out_image_path, aug_image, quality=96)
            write_yolo_label(out_label_path, aug_bboxes, aug_labels)

            written_images += 1
            pbar.update(1)

    pbar.close()
    print(f"{split_name} images written:", written_images)


def prepare_offline_aug_dataset():
    reset_dir(OFFLINE_TRAIN_IMAGES_DIR)
    reset_dir(OFFLINE_TRAIN_LABELS_DIR)
    reset_dir(OFFLINE_VALID_IMAGES_DIR)
    reset_dir(OFFLINE_VALID_LABELS_DIR)

    write_data_yaml_offline()

    materialize_augmented_split(
        image_paths=train_image_paths,
        out_images_dir=OFFLINE_TRAIN_IMAGES_DIR,
        out_labels_dir=OFFLINE_TRAIN_LABELS_DIR,
        transform=train_transform,
        split_name="train",
        copies_per_image=TRAIN_AUG_COPIES_PER_IMAGE,
        keep_original=KEEP_ORIGINAL_TRAIN_IMAGES,
    )

    materialize_augmented_split(
        image_paths=valid_image_paths,
        out_images_dir=OFFLINE_VALID_IMAGES_DIR,
        out_labels_dir=OFFLINE_VALID_LABELS_DIR,
        transform=valid_transform,
        split_name="valid",
        copies_per_image=VALID_AUG_COPIES_PER_IMAGE,
        keep_original=KEEP_ORIGINAL_VALID_IMAGES,
    )

    train_count = len(collect_images(OFFLINE_TRAIN_IMAGES_DIR))
    valid_count = len(collect_images(OFFLINE_VALID_IMAGES_DIR))

    print()
    print("Offline augmented dataset prepared")
    print("Dataset root:", OFFLINE_ROOT)
    print("Train images:", train_count)
    print("Valid images:", valid_count)

    assert train_count > 0, "В offline train/images нет картинок"
    assert valid_count > 0, "В offline valid/images нет картинок"

    return OFFLINE_DATA_YAML


OFFLINE_DATA_YAML = prepare_offline_aug_dataset()


## 13. Обучение YOLO26m

Здесь запускается обучение модели на подготовленном offline-аугментированном датасете.

Внутренние YOLO-аугментации (`mosaic`, `mixup`, `copy_paste`, HSV-сдвиги, scale/translate и т.д.) отключены специально. Это сделано для того, чтобы модель обучалась именно на контролируемых кастомных искажениях, которые были подготовлены выше.

В Colab обучение автоматически использует:

- `device=[0, 1]`, если реально доступны две GPU;
- `device=0`, если доступна одна GPU;
- `cpu`, если GPU не подключена.

После обучения сохраняются пути к лучшим весам `best.pt` и последним весам `last.pt`.


In [ ]:
# ============================================================
# 13. Train YOLO26m with internal YOLO augmentations disabled
# ============================================================

import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

assert OFFLINE_DATA_YAML.exists(), f"Не найден YAML: {OFFLINE_DATA_YAML}"

train_images_count = len(collect_images(OFFLINE_TRAIN_IMAGES_DIR))
valid_images_count = len(collect_images(OFFLINE_VALID_IMAGES_DIR))

print("OFFLINE_DATA_YAML:", OFFLINE_DATA_YAML)
print("train images:", train_images_count)
print("valid images:", valid_images_count)
print("MODEL_WEIGHTS:", MODEL_WEIGHTS)
print("IMG_SIZE:", IMG_SIZE)
print("TRAIN_EPOCHS:", TRAIN_EPOCHS)
print("TRAIN_DEVICE:", TRAIN_DEVICE)
print("BATCH_SIZE:", BATCH_SIZE)

model = YOLO(MODEL_WEIGHTS)

# Отключил все внутренние аугументации ЕЛКИ
# Модель учится на уже материализованном аугментированном датасете.
train_results = model.train(
    data=str(OFFLINE_DATA_YAML),

    epochs=TRAIN_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,

    device=TRAIN_DEVICE,
    workers=WORKERS,
    seed=SEED,
    deterministic=True,

    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,

    cache=False,
    amp=True,

    # YOLO internal augmentations OFF.
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=0,

    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,

    fliplr=0.0,
    flipud=0.0,

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    erasing=0.0,

    optimizer="AdamW",
    lr0=2e-4,
    lrf=0.05,
    weight_decay=1e-4,
    warmup_epochs=2.5,
    cos_lr=True,

    patience=8,
    save=True,
    save_period=-1,
    plots=True,
    verbose=True,
)

RUN_DIR = Path(train_results.save_dir) if train_results is not None else Path(PROJECT_DIR) / RUN_NAME
BEST_MODEL_PATH = RUN_DIR / "weights" / "best.pt"
LAST_MODEL_PATH = RUN_DIR / "weights" / "last.pt"

print("RUN_DIR:", RUN_DIR)
print("BEST_MODEL_PATH:", BEST_MODEL_PATH)
print("LAST_MODEL_PATH:", LAST_MODEL_PATH)
assert BEST_MODEL_PATH.exists(), "best.pt не найден после обучения"


## 14. Поиск test и визуализация предсказаний

Перед финальным сабмитом модель прогоняется на нескольких test-изображениях.

Эта проверка нужна, чтобы оценить качество предсказаний визуально:

- видит ли модель объекты на грязных/размытых изображениях;
- не слишком ли много лишних рамок;
- не слишком ли высокий или низкий threshold;
- адекватно ли работает NMS.

Thresholds в этой ячейке можно менять для визуального подбора поведения модели.

Что интересно, чисто визуально модель лучше отрабатывала с conf повыше, но результат лучше у заниженной уверенности ответа

In [ ]:
# ============================================================
# 14. Test resolver and prediction visualization
# ============================================================

def find_sample_submission(search_root=None):
    search_root = Path(DATASET_SEARCH_ROOT if search_root is None else search_root)
    candidates = sorted(search_root.rglob("sample_submission.csv"))
    if len(candidates) == 0:
        raise FileNotFoundError(f"sample_submission.csv не найден в {search_root}")
    return candidates[0]


def find_test_images_dir(search_root=None):
    search_root = Path(DATASET_SEARCH_ROOT if search_root is None else search_root)
    candidates = []

    for images_dir in search_root.rglob("images"):
        if not images_dir.is_dir():
            continue

        image_paths = collect_images(images_dir)
        if len(image_paths) == 0:
            continue

        path_text = str(images_dir).lower().replace("\\", "/")
        priority = 0

        if "test" in path_text:
            priority += 100
        if "train" in path_text:
            priority -= 50
        if "valid" in path_text or "val" in path_text:
            priority -= 30

        candidates.append((priority, len(image_paths), images_dir))

    if len(candidates) == 0:
        raise FileNotFoundError(f"Не удалось найти test/images в {search_root}")

    candidates = sorted(candidates, key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0][2]


SAMPLE_SUBMISSION_PATH = find_sample_submission(DATASET_SEARCH_ROOT)
TEST_IMAGES_DIR = find_test_images_dir(DATASET_SEARCH_ROOT)

sample_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
test_image_paths = collect_images(TEST_IMAGES_DIR)

test_image_map = {}
for p in test_image_paths:
    test_image_map[p.stem] = p
    test_image_map[p.name] = p

print("SAMPLE_SUBMISSION_PATH:", SAMPLE_SUBMISSION_PATH)
print("TEST_IMAGES_DIR:", TEST_IMAGES_DIR)
print("sample rows:", len(sample_df))
print("test .jpg images:", len(test_image_paths))


def get_test_path_by_image_id(image_id):
    image_id = str(image_id)

    if image_id in test_image_map:
        return test_image_map[image_id]

    stem = Path(image_id).stem
    if stem in test_image_map:
        return test_image_map[stem]

    raise KeyError(f"Не нашел test-картинку для image_id={image_id}")


best_model = YOLO(str(BEST_MODEL_PATH))

# Параметры только для визуальной проверки.
VIS_CONF_THRESHOLD = 0.05
VIS_IOU_THRESHOLD = 0.55
VIS_MAX_DET = 100
VIS_N = 12
VIS_USE_TTA = False


def visualize_test_predictions(
    model,
    n=12,
    conf_threshold=0.05,
    iou_threshold=0.55,
    max_det=100,
    use_tta=False,
    seed=SEED,
):
    rng = np.random.RandomState(seed)

    chosen_ids = rng.choice(
        sample_df["image_id"].astype(str).values,
        size=min(n, len(sample_df)),
        replace=False,
    )
    chosen_paths = [get_test_path_by_image_id(image_id) for image_id in chosen_ids]

    pred_results = model.predict(
        source=[str(p) for p in chosen_paths],
        imgsz=IMG_SIZE,
        conf=conf_threshold,
        iou=iou_threshold,
        max_det=max_det,
        augment=use_tta,
        end2end=False,
        agnostic_nms=True,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )

    for path, result in zip(chosen_paths, pred_results):
        plotted = result.plot()
        plotted_rgb = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
        n_boxes = 0 if result.boxes is None else len(result.boxes)

        plt.figure(figsize=(9, 9))
        plt.imshow(plotted_rgb)
        plt.title(f"{Path(path).name}\nconf={conf_threshold}, iou={iou_threshold}, boxes={n_boxes}")
        plt.axis("off")
        plt.show()


visualize_test_predictions(
    model=best_model,
    n=VIS_N,
    conf_threshold=VIS_CONF_THRESHOLD,
    iou_threshold=VIS_IOU_THRESHOLD,
    max_det=VIS_MAX_DET,
    use_tta=VIS_USE_TTA,
)


## 15. Финальный инференс и формирование `sample_submission.csv`

В последней части ноутбука модель предсказывает рамки для всех test-изображений и сохраняет итоговый файл сабмита в `/content/working/sample_submission.csv`.

Важные параметры:

- `SUBMIT_CONF_THRESHOLD` — минимальная уверенность предсказания;
- `SUBMIT_IOU_THRESHOLD` — IoU для NMS;
- `SUBMIT_MAX_DET` — максимум рамок на изображение;
- `SUBMIT_AGNOSTIC_NMS` — подавление дубликатов даже между разными классами;
- `SUBMIT_END2END = False` — важно для YOLO26, чтобы `iou` реально влиял на NMS.

Для инференса оставляем одну GPU (`device=0`), потому что это стабильнее в ноутбуке, а основной выигрыш от двух T4 нужен именно на обучении.


In [ ]:
# ============================================================
# 15. Full test inference and submission.csv
# ============================================================

SUBMIT_CONF_THRESHOLD = 0.001
SUBMIT_IOU_THRESHOLD = 0.65
SUBMIT_MAX_DET = 100
SUBMIT_USE_TTA = False
SUBMIT_BATCH_SIZE = 16

# Для YOLO26 важно явно выключить end-to-end режим, чтобы iou реально влиял на NMS.
SUBMIT_END2END = False

# True помогает убрать дублирующиеся рамки, даже если они разных классов.
SUBMIT_AGNOSTIC_NMS = True


def boxes_to_prediction_string(result):
    # Формат: class_id confidence x_min y_min x_max y_max ...
    if result.boxes is None or len(result.boxes) == 0:
        return ""

    boxes_xyxy = result.boxes.xyxy.detach().cpu().numpy()
    confs = result.boxes.conf.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)

    parts = []
    for cls_id, conf, box in zip(classes, confs, boxes_xyxy):
        x_min, y_min, x_max, y_max = box
        parts.extend(
            [
                str(int(cls_id)),
                f"{float(conf):.6f}",
                f"{float(x_min):.2f}",
                f"{float(y_min):.2f}",
                f"{float(x_max):.2f}",
                f"{float(y_max):.2f}",
            ]
        )

    return " ".join(parts)


ordered_image_ids = sample_df["image_id"].astype(str).tolist()
ordered_test_paths = [get_test_path_by_image_id(image_id) for image_id in ordered_image_ids]

print("Всего test изображений для submission:", len(ordered_test_paths))
print("conf:", SUBMIT_CONF_THRESHOLD)
print("iou:", SUBMIT_IOU_THRESHOLD)
print("max_det:", SUBMIT_MAX_DET)
print("tta:", SUBMIT_USE_TTA)
print("end2end:", SUBMIT_END2END)
print("agnostic_nms:", SUBMIT_AGNOSTIC_NMS)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

submission_records = []
all_box_counts = []

for start in tqdm(range(0, len(ordered_test_paths), SUBMIT_BATCH_SIZE), desc="Predict test"):
    end = start + SUBMIT_BATCH_SIZE

    batch_ids = ordered_image_ids[start:end]
    batch_paths = ordered_test_paths[start:end]

    batch_results = best_model.predict(
        source=[str(p) for p in batch_paths],
        imgsz=IMG_SIZE,
        conf=SUBMIT_CONF_THRESHOLD,
        iou=SUBMIT_IOU_THRESHOLD,
        max_det=SUBMIT_MAX_DET,
        augment=SUBMIT_USE_TTA,
        end2end=SUBMIT_END2END,
        agnostic_nms=SUBMIT_AGNOSTIC_NMS,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )

    for image_id, result in zip(batch_ids, batch_results):
        n_boxes = 0 if result.boxes is None else len(result.boxes)
        all_box_counts.append(n_boxes)

        submission_records.append(
            {
                "image_id": image_id,
                "PredictionString": boxes_to_prediction_string(result),
            }
        )

submission_df = pd.DataFrame(submission_records)
submission_df = submission_df[["image_id", "PredictionString"]]

assert len(submission_df) == len(sample_df), "Размер submission не совпадает с sample_submission"

submission_df.to_csv(SUBMISSION_PATH, index=False)

print("Готово")
print("submission shape:", submission_df.shape)
print("saved to:", SUBMISSION_PATH)

if len(all_box_counts) > 0:
    print("boxes per image:")
    print("  min:", int(np.min(all_box_counts)))
    print("  mean:", float(np.mean(all_box_counts)))
    print("  median:", float(np.median(all_box_counts)))
    print("  max:", int(np.max(all_box_counts)))

display(submission_df.head())
